# Create random raster
Here we are going to generate a random raster for testing

In [1]:
from osgeo import gdal, osr
import os
import numpy as np


In [3]:
def array2raster(Raster_file_name, originX, originY, pixelWidth, pixelHeight, raster_values, data_type, band_names=None, epsg=4326):
    """
    Create a multi-band raster from a list of arrays.
    
    Parameters:
    Raster_file_name (str): Output raster file path.
    originX (float): X-coordinate of the raster origin.
    originY (float): Y-coordinate of the raster origin.
    pixelWidth (float): Width of each pixel.
    pixelHeight (float): Height of each pixel.
    raster_values (list of numpy arrays): List of 2D arrays representing raster bands.
    data_type (GDALDataType): GDAL data type (e.g., gdal.GDT_Float32).
    band_names (list of str, optional): List of names for each band (for reference, not stored in GTiff).
    epsg (int, optional): EPSG code for spatial reference (default is 4326).
    
    Returns:
    None
    """
    
    num_bands = len(raster_values)  # Determine the number of bands based on input arrays
    rows, cols = raster_values[0].shape  # Extract dimensions from the first band

    driver = gdal.GetDriverByName('GTiff')
    outRaster = driver.Create(Raster_file_name, cols, rows, num_bands, data_type)

    # Set the geotransform
    outRaster.SetGeoTransform((originX, pixelWidth, 0, originY, 0, pixelHeight))

    # Write each band to the raster
    for i, array in enumerate(raster_values):
        reversed_arr = np.flipud(array)  # Flip array vertically
        
        outband = outRaster.GetRasterBand(i + 1)
        outband.WriteArray(reversed_arr)
        outband.FlushCache()

        if band_names and len(band_names) == num_bands:
            outband.SetDescription(band_names[i])  # Assign band names
    
    # Set projection
    outRasterSRS = osr.SpatialReference()
    outRasterSRS.ImportFromEPSG(epsg)
    outRaster.SetProjection(outRasterSRS.ExportToWkt())

    outRaster = None  # Close the file

In [4]:
if __name__ == "__main__":
    originX = -123.25745
    originY = 45.43013
    pixelWidth = 10
    pixelHeight = 10
    Raster_file_name = 'test_sample.tif'
    band_names = ["2010", "2011", "2012", "2013"] # if none, empty
    raster_values = range(1,51) # [0, 1]
    raster_size = (500, 1000)
    data_type = gdal.GDT_Byte
    
    # Generate synthetic raster bands
    num_bands = len(band_names) if band_names else 1  # If no band names, assume 1 band
    raster_arrays = [np.random.choice(raster_values, size=raster_size) for _ in range(num_bands)]

    # Call function
    array2raster(Raster_file_name, originX, originY, pixelWidth, pixelHeight, raster_arrays, data_type, band_names)